# Phase 6 · S4 (knowledge-distilled C LSTM student) generation tuning — same trained model, better decoding (no training)

The same fix that lifted S1 (validation ROUGE-1 0.461 → 0.521 with the same weights) applied to S4 (knowledge-distilled C LSTM student): it was submitted with S1's original decoding (minimum length 8% of each chunk, no repetition penalty) — Codabench BERTScore 0.8086 / ROUGE-1 0.358 / ROUGE-L 0.216; validation see S4's Cell 15.

This notebook **does not train anything**. It loads the run's trained weights and prepared chunks from its output dataset (`P6_S4_KD_A_to_C_LSTM/checkpoints/best.pt`, `P6_S4_KD_A_to_C_LSTM/data_cache.pt`) and the run's own model and cached-generation code, adds a repetition penalty (Keskar et al., 2019), and compares decoding settings on 60 held-out validation documents:
- **minimum summary length per chunk:** 15%, 22% or 30% of the chunk's tokens (maximum 50%), × **repetition penalty** 1.2 or 1.35 — around S1's best (22% / 1.2);
- plus the run's original setting (8% / 1.0) for comparison.

The best (ROUGE-1 + ROUGE-L) is re-scored on all 150 validation documents and used for the 100 test summaries. Changing only generation settings is not a new experiment — it is an improved submission of the same run.

**Before running (Kaggle)**
- Accelerator: **GPU T4** (one is enough) · internet not needed.
- Add Input → the run's output dataset **`Phase6_S4_Dataset`**, **`rohitbatra2004/phase133` (Architecture C)** and **`rohitbatra2004/cs781-phase1-tokenized-corpus`** (tokenizer).
- Expected time ≈ 45–90 minutes.

Outputs: `/kaggle/working/P6_S4_tuned/` — upload `submission.zip`.

In [ ]:
# Cell 1 — Imports, random seed and device.
# Every later cell uses the names defined here.
import os, json, glob, math, time, random, shutil, zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import hashlib, re
from sklearn.metrics import f1_score, precision_score, recall_score, classification_report, confusion_matrix
from tokenizers import Tokenizer
from huggingface_hub import snapshot_download

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch", torch.__version__, "| device:", DEVICE, "| GPUs visible:", torch.cuda.device_count())

In [ ]:
# Cell 2 — The source run's settings (needed to rebuild its model and read its data; training settings are unused here)
# and the decoding settings tuned in this notebook.
SOURCE_RUN = "P6_S4_KD_A_to_C_LSTM"                # the trained run whose checkpoint is re-decoded
RUN_NAME = "P6_S4_tuned"
TEACHER_RUN = "P6_S1_A_LORMA"
WORK_DIR = Path("/kaggle/working") / RUN_NAME
DATA_DIR = Path("/kaggle/working/data")
CKPT_DIR = WORK_DIR / "checkpoints"
OUT_DIR = WORK_DIR / "outputs"
for folder in (WORK_DIR, DATA_DIR, CKPT_DIR, OUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)
HF_REPO_ID = "Exploration-Lab/CS-781-Capstone"

# Chunk-level summarization data (prompt + summary ≤ 1,024 tokens, C's pretraining length).
CHUNK_TOKENS = 640
MAX_TARGET_TOKENS = 360
MAX_CHUNKS_PER_DOC = 70
SEPARATOR = " Summary:"             # same prompt format as the S1 teacher
VAL_DOCS = 150                      # same held-out documents as S1–S3 (same seed)

# Teacher: S1's LORMA settings (needed to rebuild it).
LORMA_RANK = 8
LORMA_ALPHA = 16.0
LORMA_TARGETS_ATTN = ("q_proj", "k_proj", "v_proj", "out_proj")
LORMA_TARGETS_FFN = ("fc1", "fc2")

# Student: Architecture C, top LSTM layer + final LayerNorm + output bias trained.
TUNED_LSTM_LAYERS = 1

# Knowledge distillation.
KD_TEMPERATURE = 2.0
KD_WEIGHT = 1.0

# Training.
EPOCHS = 2
BATCH_SEQS = 8                      # sequences per forward pass, split over both GPUs
GRAD_ACCUM = 2                      # effective batch = 16 sequences
LR = 2e-4
WEIGHT_DECAY = 0.01
WARMUP_FRACTION = 0.03
GRAD_CLIP = 1.0
SAVE_EVERY_STEPS = 1000

# Generation (greedy, cached LSTM state).
GEN_BATCH = 32
MIN_NEW_FRACTION = 0.08
MAX_NEW_FRACTION = 0.45
NO_REPEAT_NGRAM = 4
KEEP_FRACTIONS = (0.5, 0.7, 0.85, 1.0)
USE_AMP = DEVICE.type == "cuda"

# Decoding (the only thing tuned here).
MAX_NEW_FRACTION = 0.50                       # maximum summary tokens per chunk = 50% of the chunk (+ 32)
ORIGINAL_SETTING = (0.08, 1.0)                # what the run was submitted with
MIN_NEW_FRACTIONS = (0.15, 0.22, 0.30)        # minimum summary tokens per chunk
REPETITION_PENALTIES = (1.2, 1.35)            # penalty on tokens already generated for the chunk
TUNE_DOCS = 60
print("Run:", RUN_NAME, "| source:", SOURCE_RUN, "| outputs:", WORK_DIR)

In [ ]:
# Cell 3 — Load the source run's prepared data (chunks, held-out validation documents with references, test chunks) and its
# best trained weights; the data settings must equal this notebook's.
src_data = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/data_cache.pt", recursive=True))
src_ckpt = sorted(glob.glob(f"/kaggle/input/**/{SOURCE_RUN}/checkpoints/best.pt", recursive=True))
assert src_data and src_ckpt, f"Add the output dataset of {SOURCE_RUN} as an input."
data = torch.load(src_data[0], weights_only=False)
assert data["settings"]["chunk_tokens"] == CHUNK_TOKENS and data["settings"]["max_target_tokens"] == MAX_TARGET_TOKENS
best = torch.load(src_ckpt[0], map_location="cpu", weights_only=False)
print(f"{SOURCE_RUN}: {len(data['val'])} validation documents, {len(data['test'])} test documents | best epoch {best['epoch']} "
      f"(validation loss {best['val_loss']:.4f})")

In [ ]:
# Cell 4 — Tokenizer, special ids and the sentence / word patterns used by the source run (decoding and scoring).
tokenizer = Tokenizer.from_file(sorted(glob.glob("/kaggle/input/**/tokenizer/tokenizer.json", recursive=True))[0])
PAD_ID, BOS_ID, EOS_ID, UNK_ID = (tokenizer.token_to_id(t) for t in ("[PAD]", "[BOS]", "[EOS]", "[UNK]"))
assert (PAD_ID, UNK_ID, BOS_ID, EOS_ID) == (0, 1, 2, 3)
SEP_IDS = tokenizer.encode(SEPARATOR).ids
SENT_SPLIT = re.compile(r"(?<=[.?!])\s+(?=[A-Z0-9(\"'])")
WORD = re.compile(r"[a-z0-9]+")

def split_sentences(text):
    return [s for s in SENT_SPLIT.split(text) if s.strip()]
print("Tokenizer ready")

In [ ]:
# Cell 5 — Rebuild the Phase-1 Architecture C backbone and strictly load its pretrained weights.
# Student architecture. Weights from model_C_state_dict.pt; configuration from model_C_metadata.json.
c_paths = sorted(glob.glob("/kaggle/input/**/architecture_C/model_C_state_dict.pt", recursive=True))
assert c_paths, "Add the dataset rohitbatra2004/phase133 as an input."
C_ROOT = Path(c_paths[0]).parent
C_METADATA = json.load(open(C_ROOT / "model_C_metadata.json"))
C_CONFIG = C_METADATA["model_config"]
assert C_CONFIG["hidden_size"] == C_CONFIG["embedding_dim"], "Phase-1 C used an identity hidden-to-embedding map"

class ArchitectureCBackbone(nn.Module):
    """Phase-1 stacked LSTM language model without the vocabulary projection.
    Returns the normalised contextual state of every token: [batch, tokens, hidden_size]."""
    def __init__(self, cfg):
        super().__init__()
        self.token_embedding = nn.Embedding(cfg["vocab_size"], cfg["embedding_dim"])
        self.lstm = nn.LSTM(cfg["embedding_dim"], cfg["hidden_size"], cfg["num_layers"],
                            dropout=cfg["dropout"], bias=cfg["use_bias"], batch_first=True)
        self.final_norm = nn.LayerNorm(cfg["hidden_size"])
        # Kept only so that the saved Phase-1 state dict loads strictly; unused here.
        self.output_bias = nn.Parameter(torch.zeros(cfg["vocab_size"]))

    def forward(self, input_ids):
        states, _ = self.lstm(self.token_embedding(input_ids))
        return self.final_norm(states)

backbone = ArchitectureCBackbone(C_CONFIG)
saved = torch.load(c_paths[0], map_location="cpu", weights_only=True)
# The export is either the raw state dict or a dict that wraps it under "state_dict".
phase1_state = saved["state_dict"] if "state_dict" in saved else saved
backbone.load_state_dict(phase1_state, strict=True)

n_backbone = sum(p.numel() for p in backbone.parameters())
assert n_backbone == int(C_METADATA["parameter_count"]), "Parameter count differs from Phase-1 metadata"
print(json.dumps(C_CONFIG, indent=2))
print(f"Loaded Phase-1 Architecture C from {C_ROOT} — {n_backbone:,} parameters")

In [ ]:
# Cell 6 — Student set-up: train the top LSTM layer, final LayerNorm and output bias; embeddings and lower LSTM layers
# stay as pretrained. Then the distillation model: student + frozen teacher on the same sequence.
for p in backbone.parameters():
    p.requires_grad_(False)
tuned_layers = [C_CONFIG["num_layers"] - 1 - i for i in range(TUNED_LSTM_LAYERS)]
for name, p in backbone.named_parameters():
    if any(name.endswith(f"_l{layer}") for layer in tuned_layers) or name.startswith(("final_norm", "output_bias")):
        p.requires_grad_(True)
adapted_backbone = backbone.to(DEVICE)
HIDDEN = C_CONFIG["hidden_size"]

# Check on the device: only the chosen layer and the final LayerNorm receive gradients.
adapted_backbone(torch.randint(4, 1000, (2, 16), device=DEVICE)).float().pow(2).mean().backward()
tuned = [n for n, p in adapted_backbone.named_parameters() if p.requires_grad]
assert all(adapted_backbone.get_parameter(n).grad is not None for n in tuned if n != "output_bias"), "A tuned parameter got no gradient"
assert all(p.grad is None for p in adapted_backbone.parameters() if not p.requires_grad), "A frozen parameter got a gradient"
adapted_backbone.zero_grad(set_to_none=True)
n_tuned = sum(p.numel() for p in adapted_backbone.parameters() if p.requires_grad)
print(f"Tuned LSTM layer(s) {tuned_layers} + final LayerNorm: {tuned}")
print(f"{n_tuned:,} trainable backbone parameters ({100 * n_tuned / n_backbone:.2f}% of the backbone)")

class StudentLM(nn.Module):
    """The trained C student: final_norm states × tied embedding + output bias."""
    def __init__(self, student):
        super().__init__()
        self.backbone = student

model = StudentLM(adapted_backbone).to(DEVICE)
SAVED_NAMES = [n for n, p in model.named_parameters() if p.requires_grad]
print(f"Student ready: {sum(p.numel() for p in model.parameters()):,} parameters")

In [ ]:
# Cell 7 — Load the source run's trained weights into the rebuilt model (the parameter names must match exactly).
assert set(best["trainable"]) == set(SAVED_NAMES), "Checkpoint does not match this run's trainable parameters"
model.load_state_dict(best["trainable"], strict=False)
model.eval()
print(f"Loaded {len(SAVED_NAMES)} trained tensors from {src_ckpt[0]}")

In [ ]:
# Cell 8 — (the source run's generation code, plus an optional repetition penalty) Cached greedy generation for the LSTM student: the prompt is run once (packed, so padding never enters the
# state), then one LSTM step per new token from the carried (h, c) state. Repeated 4-grams blocked, min / max lengths.
def block_repeats(logits, generated, n):
    """Forbid tokens that would repeat an n-gram already generated in that row."""
    for b, seq in enumerate(generated):
        if len(seq) >= n - 1:
            prefix = tuple(seq[-(n - 1):])
            banned = [seq[i + n - 1] for i in range(len(seq) - n + 1) if tuple(seq[i:i + n - 1]) == prefix]
            if banned:
                logits[b, banned] = -1e4

@torch.no_grad()
def generate(bb, prompts, max_new, min_new, repetition_penalty=1.0):
    bb.eval()
    B, L = len(prompts), max(len(p) for p in prompts)
    ids = torch.full((B, L), PAD_ID, dtype=torch.long, device=DEVICE)
    for b, p in enumerate(prompts):
        ids[b, :len(p)] = torch.tensor(p, device=DEVICE)
    lengths = torch.tensor([len(p) for p in prompts])
    emb, bias = bb.token_embedding.weight, bb.output_bias
    with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=USE_AMP):
        packed = nn.utils.rnn.pack_padded_sequence(bb.token_embedding(ids), lengths, batch_first=True, enforce_sorted=False)
        out, state = bb.lstm(packed)
        out, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True)
        last = out[torch.arange(B, device=DEVICE), lengths.to(DEVICE) - 1]
        logits = F.linear(bb.final_norm(last), emb, bias).float()
        generated, done = [[] for _ in range(B)], [False] * B
        for step in range(max(max_new)):
            logits[:, [PAD_ID, BOS_ID, UNK_ID]] = -1e4
            for b in range(B):
                if step < min_new[b]:
                    logits[b, EOS_ID] = -1e4
            if repetition_penalty != 1.0:                         # CTRL-style penalty on tokens already generated
                for b, seq in enumerate(generated):
                    if seq:
                        idx = torch.tensor(sorted(set(seq)), device=DEVICE)
                        v = logits[b, idx]
                        logits[b, idx] = torch.where(v > 0, v / repetition_penalty, v * repetition_penalty)
            block_repeats(logits, generated, NO_REPEAT_NGRAM)
            token = logits.argmax(-1)
            for b in range(B):
                if not done[b]:
                    if token[b].item() == EOS_ID or len(generated[b]) >= max_new[b]:
                        done[b] = True
                    else:
                        generated[b].append(token[b].item())
            if all(done):
                break
            out, state = bb.lstm(bb.token_embedding(token[:, None]), state)
            logits = F.linear(bb.final_norm(out[:, -1]), emb, bias).float()
    return generated

def summarise(items, min_fraction, repetition_penalty, first_chunk_only=False):
    """Per document: list of per-chunk summaries (each a list of sentences)."""
    jobs = [(i, j, c) for i, d in enumerate(items) for j, c in enumerate(d["chunks"][:1] if first_chunk_only else d["chunks"])]
    jobs.sort(key=lambda job: len(job[2]))
    out = [dict() for _ in items]
    for s in range(0, len(jobs), GEN_BATCH):
        batch = jobs[s:s + GEN_BATCH]
        prompts = [[BOS_ID] + c.tolist() + SEP_IDS for _, _, c in batch]
        max_new = [min(MAX_TARGET_TOKENS, int(MAX_NEW_FRACTION * len(c)) + 32) for _, _, c in batch]
        min_new = [int(min_fraction * len(c)) for _, _, c in batch]
        for (i, j, _), g in zip(batch, generate(model.backbone, prompts, max_new, min_new, repetition_penalty)):
            out[i][j] = split_sentences(tokenizer.decode(g).strip())
    return [[out[i][j] for j in sorted(out[i])] for i in range(len(items))]

def join_summary(chunk_summaries, keep_fraction=1.0):
    """Keep the first share of each chunk summary's sentences, drop sentences already used, join in document order."""
    seen, kept = set(), []
    for sentences in chunk_summaries:
        for s in sentences[:max(1, math.ceil(keep_fraction * len(sentences)))] if sentences else []:
            key = " ".join(WORD.findall(s.lower()))
            if key and key not in seen:
                seen.add(key); kept.append(s)
    return " ".join(kept)
print("Generation helpers ready.")

In [ ]:
# Cell 9 — ROUGE-1 / ROUGE-L F1 (word level, lower-cased) and repetition, for the held-out validation documents.
def rouge1_f(ref, hyp):
    overlap = sum((Counter(ref) & Counter(hyp)).values())
    return 0.0 if overlap == 0 else 2 * overlap / (len(ref) + len(hyp))

def rougeL_f(ref, hyp, cap=800):
    ref, hyp = ref[:cap], hyp[:cap]
    prev = [0] * (len(hyp) + 1)
    for x in ref:
        cur = [0]
        for j, y in enumerate(hyp, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[j - 1]))
        prev = cur
    return 0.0 if prev[-1] == 0 else 2 * prev[-1] / (len(ref) + len(hyp))

def score_summaries(refs, hyps):
    r1, rl, lengths, rep = [], [], [], []
    for ref, hyp in zip(refs, hyps):
        rw, hw = WORD.findall(ref.lower()), WORD.findall(hyp.lower())
        r1.append(rouge1_f(rw, hw)); rl.append(rougeL_f(rw, hw)); lengths.append(len(hw) / max(len(rw), 1))
        grams = list(zip(hw, hw[1:], hw[2:]))
        rep.append(1 - len(set(grams)) / max(len(grams), 1))
    return {"rouge1": float(np.mean(r1)), "rougeL": float(np.mean(rl)), "length_ratio": float(np.median(lengths)),
            "repeated_trigram_share": float(np.mean(rep))}

def describe(s):
    return (f"ROUGE-1 {s['rouge1']:.4f} | ROUGE-L {s['rougeL']:.4f} | length vs reference ×{s['length_ratio']:.2f} | "
            f"repeated trigrams {100 * s['repeated_trigram_share']:.1f}%")
print("Scoring ready (validation only).")

In [ ]:
# Cell 10 — Compare the decoding settings on TUNE_DOCS validation documents (keep all sentences of every chunk summary);
# the first row is the setting the run was submitted with.
tune_items = data["val"][:TUNE_DOCS]
tune_refs = [d["summary"] for d in tune_items]
grid = {}
for min_fraction, penalty in [ORIGINAL_SETTING] + [(m, p) for m in MIN_NEW_FRACTIONS for p in REPETITION_PENALTIES]:
    start_time = time.time()
    summaries = [join_summary(c) for c in summarise(tune_items, min_fraction, penalty)]
    grid[(min_fraction, penalty)] = score_summaries(tune_refs, summaries)
    print(f"  min length {min_fraction:.2f} · repetition penalty {penalty:.2f}: {describe(grid[(min_fraction, penalty)])} "
          f"| {(time.time() - start_time) / 60:.1f} min", flush=True)
BEST_MIN, BEST_PENALTY = max(grid, key=lambda k: grid[k]["rouge1"] + grid[k]["rougeL"])
print(f"Chosen: minimum length {BEST_MIN:.2f} of each chunk, repetition penalty {BEST_PENALTY:.2f} "
      f"(original {ORIGINAL_SETTING}: {grid[ORIGINAL_SETTING]['rouge1']:.4f} / {grid[ORIGINAL_SETTING]['rougeL']:.4f})")

In [ ]:
# Cell 11 — Re-score the chosen setting on all held-out validation documents (comparable with the run's own validation score).
refs = [d["summary"] for d in data["val"]]
start_time = time.time()
val_chunk_summaries = summarise(data["val"], BEST_MIN, BEST_PENALTY)
val_result = score_summaries(refs, [join_summary(c) for c in val_chunk_summaries])
print(f"All {len(refs)} validation documents: {describe(val_result)} | {(time.time() - start_time) / 60:.1f} min")
print("\nExample (validation) — reference:\n", data["val"][0]["summary"][:700],
      "\n\nGenerated:\n", join_summary(val_chunk_summaries[0])[:700])

In [ ]:
# Cell 12 — Test summaries with the chosen setting, format checks, submission.zip and a run summary.
start_time = time.time()
test_chunk_summaries = summarise(data["test"], BEST_MIN, BEST_PENALTY)
prediction = [{"id": d["id"], "summary": join_summary(c)} for d, c in zip(data["test"], test_chunk_summaries)]
for p, d in zip(prediction, data["test"]):
    if not p["summary"].strip():
        p["summary"] = tokenizer.decode(d["chunks"][0][:200].tolist())
assert len({p["id"] for p in prediction}) == len(prediction) == len(data["test"])
assert all(isinstance(p["summary"], str) and p["summary"].strip() for p in prediction)
json.dump(prediction, open(WORK_DIR / "prediction.json", "w"), indent=1)
with zipfile.ZipFile(WORK_DIR / "submission.zip", "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(WORK_DIR / "prediction.json", arcname="prediction.json")
with zipfile.ZipFile(WORK_DIR / "submission.zip") as zf:
    assert zf.namelist() == ["prediction.json"]
words = np.array([len(p["summary"].split()) for p in prediction])
json.dump({"source": SOURCE_RUN, "min_new_fraction": BEST_MIN, "repetition_penalty": BEST_PENALTY, "max_new_fraction": MAX_NEW_FRACTION,
           "grid": {f"{k[0]}_{k[1]}": v for k, v in grid.items()}, "validation_all": val_result,
           "test_words_median": int(np.median(words))}, open(OUT_DIR / "run_summary.json", "w"), indent=2)
print(f"submission.zip: {len(prediction)} summaries | words per summary median {int(np.median(words))} (references ≈ 640) "
      f"| generation {(time.time() - start_time) / 60:.1f} min")